# Manual Test: File Lock Warning Dialog

This notebook guides you through manually testing the file-lock warning dialog that appears
when a user opens a notebook that is already locked by another user.

**How locking works:**
- Only notebooks inside `Projects/` or `Personal/` folders are subject to file locking.
- The first user to open a notebook acquires the lock and gets read-write access.
- Any subsequent user who opens the same notebook gets read-only access and sees a warning dialog.
- Locks are kept alive via a heartbeat every 10 seconds and expire after 30 seconds of inactivity.

**What this test verifies:**
1. The warning dialog appears when the file is locked.
2. The dialog shows the correct locking user's name.
3. The notebook opens in read-only mode (toolbar shows "Read Only" label).
4. Edits made in read-only mode are not saved to disk.

## Step 1: Setup — Create a Test Notebook

Run the cell below to create a fresh test notebook at `Projects/test_file_lock_dialog.ipynb`.
The `Projects/` folder is one of the two folders where file locking is enforced.

In [ ]:
import json
import os
from pathlib import Path

import jupyter_core.paths

# The root directory that Jupyter Server serves files from.
root_dir = Path(jupyter_core.paths.jupyter_runtime_dir()).parent
# In practice this is usually the directory JupyterLab was launched from.
# Adjust if your setup is different:
root_dir = Path(os.getcwd())
print(f"Jupyter root dir: {root_dir}")

target_dir = root_dir / "Projects"
target_dir.mkdir(parents=True, exist_ok=True)

test_nb_path = target_dir / "test_file_lock_dialog.ipynb"

# Minimal valid notebook.
stub_notebook = {
    "nbformat": 4,
    "nbformat_minor": 5,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {"name": "python", "version": "3.11"}
    },
    "cells": [
        {
            "cell_type": "markdown",
            "id": "test-cell",
            "metadata": {},
            "source": ["# File Lock Dialog Test Notebook\n",
                       "\n",
                       "This notebook is used for manually testing the file lock warning dialog.\n",
                       "Try editing this cell — if you are in read-only mode your changes will not be saved."]
        }
    ]
}

with open(test_nb_path, "w") as f:
    json.dump(stub_notebook, f, indent=1)

print(f"Created: {test_nb_path}")

## Step 2: Open the Test Notebook as User A (Lock Owner)

1. In the **current** JupyterLab window, open the file browser and navigate to `Projects/`.
2. Double-click **`test_file_lock_dialog.ipynb`** to open it.
3. Wait a few seconds for the WebSocket connection to establish — User A now holds the lock.

> **Tip:** You can confirm the lock was acquired by running **Step 3** below in a separate
> browser tab or by using the `lock_manager.ipynb` notebook.

## Step 3: Verify the Lock Was Acquired

Run the cell below (while the test notebook is open) to check that the lock is held.

In [ ]:
import sqlite3
import time
from datetime import datetime

# Default DB path — change to match your jupyter_server_ydoc configuration if different.
DB_PATH = "/srv/collaboration/collaboration_locks.db"

# Lock key format used by the server.
# For a notebook at Projects/test_file_lock_dialog.ipynb the key is:
EXPECTED_LOCK_KEY = "notebook:Projects/test_file_lock_dialog.ipynb"


def show_locks(db_path: str) -> None:
    """Print all active (non-expired) locks from the SQLite DB."""
    try:
        con = sqlite3.connect(db_path)
        con.row_factory = sqlite3.Row
        now = time.time()
        ttl = 30  # Default TTL — must match server config.
        rows = con.execute(
            "SELECT * FROM doc_locks WHERE (? - heartbeat_at) <= ? ORDER BY acquired_at",
            (now, ttl)
        ).fetchall()
        con.close()

        if not rows:
            print("No active locks.")
            return

        print(f"{'Lock Key':<55} {'Owner':<20} {'Age (s)':>8} {'Conns':>6}")
        print("-" * 95)
        for row in rows:
            age = now - row["acquired_at"]
            print(
                f"{row['lock_key']:<55} {row['owner']:<20} {age:>8.1f} {row['connections']:>6}"
            )
    except Exception as e:
        print(f"Could not read lock DB: {e}")


show_locks(DB_PATH)

## Step 4: Open the Test Notebook as User B (Triggers Dialog)

You need a **second authenticated session** with a different username. Use one of:

- An **incognito / private-browsing window** (logs in as a different user).
- A **second browser** (e.g. Chrome + Firefox).
- A second JupyterLab instance running on a different port.

Steps:
1. In the second session, navigate to `Projects/test_file_lock_dialog.ipynb`.
2. Open the notebook.

**Expected result:**
- A modal dialog appears with title **"Read-Only Mode"**.
- The body reads: *"File currently in use by \<USER A NAME\>; opened in read-only mode. …"*
- After clicking **Continue**, the notebook opens with a **"Read Only"** badge in the toolbar.

**What to check if no dialog appears:**
- Confirm the notebook is inside `Projects/` (not at the root or in another folder).
- Confirm User A's session is still open (lock may have expired if >30 s with no activity).
- Check the browser console for WebSocket errors.

## Step 5: Verify Read-Only Behaviour

While User B has the notebook open in read-only mode:

| Action | Expected Behaviour |
|--------|--------------------|
| Edit a cell | Cell content changes locally but **not** persisted |
| Press `Ctrl+S` / Save button | Save silently fails (no error, no disk write) |
| Toolbar | Shows **"Read Only"** label next to kernel name |
| Timeline slider (if enabled) | Shows notification: *"cannot be used in read-only mode"* |

Run the cell below to confirm the file on disk has not been modified.

In [ ]:
import time
from pathlib import Path

nb_path = Path(os.getcwd()) / "Projects" / "test_file_lock_dialog.ipynb"
mtime = nb_path.stat().st_mtime
print(f"Last modified: {datetime.fromtimestamp(mtime).isoformat(timespec='seconds')}")
print("(Make an edit + save in User B's session, then re-run this cell.")
print(" The timestamp should NOT advance while User B is in read-only mode.)")

## Step 6: Test Lock Expiry / Transfer

To test that the lock is released when User A closes the notebook:

1. Close the test notebook in **User A's** session (close the tab).
2. Wait ~5 seconds for the WebSocket to close and the lock to be released.
3. Close and **re-open** the same notebook in **User B's** session.
4. **Expected:** No dialog this time — User B now acquires the lock and gets read-write access.

Alternatively, use `lock_manager.ipynb` to forcefully release the lock without closing User A's session.

## Step 7: Cleanup

In [ ]:
import shutil

test_nb = Path(os.getcwd()) / "Projects" / "test_file_lock_dialog.ipynb"
if test_nb.exists():
    test_nb.unlink()
    print(f"Deleted {test_nb}")
else:
    print("Test notebook not found (already deleted?).")